In [ ]:
import Pkg;
Pkg.add("StatsBase")

using LinearAlgebra
using StatsBase

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
    Updating `~/.julia/environments/v1.11/Project.toml`
  [2913bbd2] + StatsBase v0.34.10
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [ ]:
using Random, StatsBase,SparseArrays

Random.seed!(1234)  # set the global seed

TaskLocalRNG()

In [ ]:
function cliqueSampleMultiEdgeSddEff(a::Vector{ComplexF64}, multiplicity::Vector{Int}, i::Int)
    n = length(a)

    # 1. Pre-calculate magnitudes to avoid repeated abs() calls
    # We copy 'a' logic without mutating the input vector permanently
    abs_a = abs.(a)
    val_i = a[i]
    abs_val_i = abs_a[i]

    # 2. Logic for self_loop and initialization
    # Original logic: self_loop = 2*|a[i]| - sum(|a|)
    # Note: sum(abs_a) includes |a[i]|, so we use that.
    total_weight = abs_val_i
    sum_abs_all = sum(abs_a)
    self_loop = abs(2 * abs_val_i - sum_abs_all)
    #println(self_loop)
    # 3. Initialize Output Matrices
    cliqueSampleLap = zeros(ComplexF64, n, n)
    new_multiedges = zeros(Int, n, n)

    # 4. Handle Diagonal Initialization (Vectorized)
    # Corresponds to: cliqueSampleLap[diagind] .+= self_loop/total_weight * abs.(a)
    # In original code, a[i] was set to 0 before this, so index i gets 0 update.
    ratio = self_loop / total_weight
    @inbounds for k in 1:n
        if k != i
            cliqueSampleLap[k, k] = ratio * abs_a[k]
        end
    end

    # 5. Setup Sampling ONCE (Big Performance Win)
    # We mask position i to 0.0 so we never sample it (removes need for k != i check)
    sampling_weights = copy(abs_a)
    sampling_weights[i] = self_loop
    wv = Weights(sampling_weights) # O(N) operation happens only once now

    # Identify neighbors (indices where weight > threshold)
    # We use valid indices from abs_a (excluding i effectively because we skip it in loop or weights)
    nbrs = findall(x -> x > 1e-12, sampling_weights)

    # 6. Main Loop
    for j in nbrs
        # Pre-calculate j-specific constants
        val_j = a[j]
        abs_val_j = abs_a[j]
        sign_j = sign(a[j]) #val_j / abs_val_j # Equivalent to sign(a[j]) for non-zero
        mult_j = multiplicity[j]

        # Scaling factor from logic: multiWeightJ = a[j] / multiplicity[j]
        # We need this for the wjk calc
        val_j_scaled = val_j / mult_j

        for rep in 1:mult_j
            # O(1) Sampling
            k = sample(wv)

            # We guaranteed k != i via weights. We only check k != j.
            if (k != j) & (j != i) & (k != i)
                val_k = a[k]
                abs_val_k = abs_a[k]

                # Calculate Weight wjk
                # wjk = abs(multiWeightJ * a[k]) / (|a[j]| + |a[k]|)
                wjk = abs(val_j_scaled * val_k) / (abs_val_j + abs_val_k)

                # Update Edge Counts
                @inbounds new_multiedges[j, k] += 1
                @inbounds new_multiedges[k, j] += 1

                # 7. Rank-1 Matrix Update (Scalar optimization)
                # Instead of L += w * b * b', we update 4 indices manually.
                # b[j] = 1, b[k] = -sign(a[k])/sign(a[j])
                s = -(val_k / abs_val_k) / sign_j # s is the value at b[k]

                # The outer product adds:
                # [j,j] -> |1|^2 * w      [j,k] -> 1 * conj(s) * w
                # [k,j] -> s * 1 * w      [k,k] -> |s|^2 * w

                # Note: |s| = 1 for complex sign, so |s|^2 = 1.

                @inbounds begin
                    cliqueSampleLap[j, j] += wjk
                    cliqueSampleLap[k, k] += wjk
                    cliqueSampleLap[j, k] += wjk * conj(s)
                    cliqueSampleLap[k, j] += wjk * s
                end
            end
        end
    end

    return cliqueSampleLap, new_multiedges
end

cliqueSampleMultiEdgeSddEff (generic function with 1 method)

In [ ]:
function initMultiedges(L, nr_multi)
  n = size(L)[1]
  multiedges = zeros(Int, n,n,)
  A_mask = abs.(-L + Diagonal(diag(L))).> 0
  multiedges[A_mask] .= nr_multi
  return multiedges
end

initMultiedges (generic function with 1 method)

In [ ]:
# This is still the same algorithm and some index handling
function apxCholMultiSDD(L, nr_multi)
  multi_edges = initMultiedges(L, nr_multi)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(ComplexF64,n,n)
  for i = 1:(n-1)

    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    ai = S[:,i]
    di = S[i,i]

    # S without star on i
    S_noi = copy(S)
    S_noi -=Diagonal(abs.(ai))
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    sampled_clique, new_multiedges = cliqueSampleMultiEdgeSddEff(ai, multi_edges[:,i],i)
    S = S_noi + sampled_clique
    multi_edges += new_multiedges
    if i==n-1
      fac[n,n] = sqrt(abs(S_noi[n,n]+ sampled_clique[n,n]))
    end
  end
  return fac
end

apxCholMultiSDD (generic function with 1 method)

In [ ]:
function generate_hermitian_dominant(n::Int; strict::Bool=true)
    # 1. Generate a random complex matrix
    A = rand(ComplexF64, n, n)

    # 2. Make it Hermitian: H = A + A'
    # Note: The diagonal of (A + A') is guaranteed to be real because z + conj(z) = 2*Real(z)
    H = A + A'

    # 3. Adjust Diagonal to enforce Dominance
    for i in 1:n
        # Calculate sum of absolute values of off-diagonal elements in row i
        off_diag_sum = sum(abs, H[i, :]) - abs(H[i, i])

        # Determine the buffer to add (random positive value for strictness)
        buffer = strict ? (0.1 + rand()) : 0.0

        # Set the diagonal element. It must be real to remain Hermitian.
        H[i, i] = off_diag_sum + buffer
    end

    return H
end

generate_hermitian_dominant (generic function with 1 method)

In [ ]:
for i in 1:10
  n = 100
  S = generate_hermitian_dominant(n)
  fApxChol = apxCholMultiSDD(S, 100) # Second param specifies the number of multiedges
  rel_eigs = eigvals(S-fApxChol*fApxChol')

  # The ratio of the smallest to the largest eigenvalue

  println(maximum(abs.(rel_eigs)) / minimum(abs.(rel_eigs)))
end

1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0


In [ ]:
function cliqueSampleMultiEdgeSddEffSparse(a::SparseVector{ComplexF64, Int}, multiplicity::SparseVector{Int, Int}, i::Int)
    n = length(a)

    # 1. Work with stored values directly (O(degree) instead of O(N))
    # 'rows' contains the indices of neighbors (and likely 'i' itself)
    rows = a.nzind
    vals = a.nzval

    # Pre-calculate magnitudes for stored entries only
    # This creates a small vector of size nnz(a), not N
    abs_vals = abs.(vals)

    # 2. Find index of 'i' in the sparse storage
    # SparseVectors are sorted, so we can use searchsortedfirst
    ptr_i = searchsortedfirst(rows, i)

    # Check if 'i' is actually stored in the sparse vector
    has_i = (ptr_i <= length(rows)) && (rows[ptr_i] == i)

    val_i = has_i ? vals[ptr_i] : 0.0 + 0.0im
    abs_val_i = has_i ? abs_vals[ptr_i] : 0.0

    # 3. Logic for self_loop
    # sum(abs_vals) sums only the stored non-zeros
    sum_abs_all = sum(abs_vals)
    self_loop = abs(2 * abs_val_i - sum_abs_all)

    # 4. Initialize Output Sparse Matrices
    # We calculate capacity to avoid resizing
    # Note: accessing sparse multiplicity[row] is O(log nnz), which is acceptable
    total_reps = 0
    @inbounds for (idx, r) in enumerate(rows)
        if r != i
            # If multiplicity is sparse, ensure we handle missing keys (assumed 0 or 1 depending on logic)
            # Assuming multiplicity is consistent with 'a' structure:
            total_reps += multiplicity[r]
        end
    end

    # Pre-allocate output triplets
    I_lap = Int[]; sizehint!(I_lap, n + 4*total_reps)
    J_lap = Int[]; sizehint!(J_lap, n + 4*total_reps)
    V_lap = ComplexF64[]; sizehint!(V_lap, n + 4*total_reps)

    I_edge = Int[]; sizehint!(I_edge, 2*total_reps)
    J_edge = Int[]; sizehint!(J_edge, 2*total_reps)
    V_edge = Int[]; sizehint!(V_edge, 2*total_reps)

    # 5. Handle Diagonal Initialization
    # We only iterate over existing non-zeros (neighbors)
    total_weight = abs_val_i
    ratio = (total_weight > 1e-12) ? self_loop / total_weight : 0.0

    @inbounds for (idx, k) in enumerate(rows)
        if k != i
            push!(I_lap, k); push!(J_lap, k); push!(V_lap, ratio * abs_vals[idx])
        end
    end

    # 6. Setup Compressed Sampling
    # We create weights for the 'rows' array, not the full 1:n range.
    sampling_weights = copy(abs_vals)
    if has_i
        sampling_weights[ptr_i] = self_loop
    else
        # If 'i' wasn't in sparse structure, we technically can't sample it easily
        # without resizing arrays. But usually for Laplacians, diagonal 'i' is always present.
        # If strictly necessary, one would push!(rows, i) and push!(sampling_weights, self_loop), re-sort.
        # Assuming 'i' exists in 'a' for SDD matrices:
        error("Diagonal entry a[i] is missing from sparse structure")
    end

    wv = Weights(sampling_weights) # Size is degree(i), very small

    # 7. Main Loop
    # Iterate over indices into the compressed vectors 'rows'/'vals'
    for (j_idx, j) in enumerate(rows)
        # Skip if weight is negligible or it is 'i'
        if j == i || sampling_weights[j_idx] <= 1e-12
            continue
        end

        val_j = vals[j_idx]
        abs_val_j = abs_vals[j_idx]
        inv_sign_j = conj(val_j / abs_val_j)

        mult_j = multiplicity[j] # Sparse lookup
        val_j_scaled = val_j / mult_j

        for rep in 1:mult_j
            # O(1) Sampling from compressed vector
            # This returns an index into 'rows', not the node index directly
            k_idx = sample(wv)
            k = rows[k_idx]

            if k != j && k != i
                val_k = vals[k_idx]
                abs_val_k = abs_vals[k_idx]

                # Weight calculation
                wjk = abs(val_j_scaled * val_k) / (abs_val_j + abs_val_k)

                # Update Edges
                push!(I_edge, j); push!(J_edge, k); push!(V_edge, 1)
                push!(I_edge, k); push!(J_edge, j); push!(V_edge, 1)

                # Update Laplacian (Rank-1)
                s = -(val_k / abs_val_k) * inv_sign_j

                push!(I_lap, j); push!(J_lap, j); push!(V_lap, wjk)
                push!(I_lap, k); push!(J_lap, k); push!(V_lap, wjk)
                push!(I_lap, j); push!(J_lap, k); push!(V_lap, wjk * conj(s))
                push!(I_lap, k); push!(J_lap, j); push!(V_lap, wjk * s)
            end
        end
    end

    # Construct final matrices
    cliqueSampleLap = sparse(I_lap, J_lap, V_lap, n, n)
    new_multiedges = sparse(I_edge, J_edge, V_edge, n, n)

    return cliqueSampleLap, new_multiedges
end

cliqueSampleMultiEdgeSddEffSparse (generic function with 1 method)

In [ ]:
function cliqueSampleMultiEdgeSddEffSparseOpt(a::SparseVector{ComplexF64, Int}, multiplicity::SparseVector{Int, Int}, i::Int)
    n = length(a)

    # Access raw storage directly
    rows = a.nzind
    vals = a.nzval
    m = length(rows) # Degree of i (number of stored entries)

    # 1. Pre-compute magnitudes and signs (Vectorized)
    # avoiding sqrt and division inside the hot loop
    abs_vals = abs.(vals)
    signs = vals ./ abs_vals

    # 2. Efficiently find index of diagonal 'i'
    ptr_i = searchsortedfirst(rows, i)
    has_i = (ptr_i <= m) && (rows[ptr_i] == i)

    abs_val_i = has_i ? abs_vals[ptr_i] : 0.0

    # 3. Calculate Self Loop
    sum_abs_all = sum(abs_vals)
    self_loop = abs(2 * abs_val_i - sum_abs_all)

    # 4. Prepare Sampling Weights
    # We copy abs_vals because we need to modify the weight for 'i'
    sampling_weights = copy(abs_vals)
    if has_i
        sampling_weights[ptr_i] = self_loop
    else
        error("Diagonal entry a[i] is missing from sparse structure")
    end

    wv = Weights(sampling_weights)

    # 5. Fast Multiplicity Lookup
    # Extract multiplicities into a dense vector aligned with 'rows'.
    # This prevents O(log N) lookups inside the hot loop.
    node_mults = Vector{Int}(undef, m)
    total_reps = 0

    @inbounds for k in 1:m
        if k == ptr_i
            node_mults[k] = 0
        else
            # Accessing sparse vector here is okay as it is done only 'm' times total
            mu = multiplicity[rows[k]]
            node_mults[k] = mu
            total_reps += mu
        end
    end

    # 6. Pre-allocate Output Arrays (Exact or Upper Bound)
    # Laplacian: (neighbors) diagonal entries + 4 * total_reps (rank-1 updates)
    # Edge: 2 * total_reps
    num_neighbors = has_i ? m - 1 : m
    max_lap_entries = num_neighbors + 4 * total_reps
    max_edge_entries = 2 * total_reps

    I_lap = Vector{Int}(undef, max_lap_entries)
    J_lap = Vector{Int}(undef, max_lap_entries)
    V_lap = Vector{ComplexF64}(undef, max_lap_entries)

    I_edge = Vector{Int}(undef, max_edge_entries)
    J_edge = Vector{Int}(undef, max_edge_entries)
    V_edge = Vector{Int}(undef, max_edge_entries)

    # 7. Diagonal Initialization
    idx_lap = 1
    total_weight = abs_val_i
    ratio = (total_weight > 1e-12) ? self_loop / total_weight : 0.0

    @inbounds for k in 1:m
        if k != ptr_i
            I_lap[idx_lap] = rows[k]
            J_lap[idx_lap] = rows[k]
            V_lap[idx_lap] = ratio * abs_vals[k]
            idx_lap += 1
        end
    end

    # 8. Main Loop
    idx_edge = 1

    @inbounds for j_idx in 1:m
        # Skip 'i' or negligible weights
        if j_idx == ptr_i || sampling_weights[j_idx] <= 1e-12
            continue
        end

        # Load j properties from pre-computed vectors
        row_j = rows[j_idx]
        val_j = vals[j_idx]
        abs_val_j = abs_vals[j_idx]
        inv_sign_j = conj(signs[j_idx])

        mult_j = node_mults[j_idx]
        val_j_scaled = val_j / mult_j

        for rep in 1:mult_j
            # Sample k index from compressed list
            k_idx = sample(wv)

            # Skip invalid samples
            if k_idx != j_idx && k_idx != ptr_i
                row_k = rows[k_idx]
                val_k = vals[k_idx]
                abs_val_k = abs_vals[k_idx]

                # Weight Calculation
                wjk = abs(val_j_scaled * val_k) / (abs_val_j + abs_val_k)

                # Laplacian Rank-1 Update components
                # s = -(val_k / abs_val_k) * inv_sign_j
                s = -(signs[k_idx]) * inv_sign_j

                val_diag = wjk
                val_off1 = wjk * conj(s)
                val_off2 = wjk * s

                # Fill Edge Buffers
                I_edge[idx_edge] = row_j; J_edge[idx_edge] = row_k; V_edge[idx_edge] = 1
                idx_edge += 1
                I_edge[idx_edge] = row_k; J_edge[idx_edge] = row_j; V_edge[idx_edge] = 1
                idx_edge += 1

                # Fill Laplacian Buffers
                I_lap[idx_lap] = row_j; J_lap[idx_lap] = row_j; V_lap[idx_lap] = val_diag
                idx_lap += 1
                I_lap[idx_lap] = row_k; J_lap[idx_lap] = row_k; V_lap[idx_lap] = val_diag
                idx_lap += 1
                I_lap[idx_lap] = row_j; J_lap[idx_lap] = row_k; V_lap[idx_lap] = val_off1
                idx_lap += 1
                I_lap[idx_lap] = row_k; J_lap[idx_lap] = row_j; V_lap[idx_lap] = val_off2
                idx_lap += 1
            end
        end
    end

    # 9. Trim arrays
    # Since we skipped some samples (k == j || k == i), we might have unused space.
    # resize! is cheap (just updates length property).
    resize!(I_lap, idx_lap - 1)
    resize!(J_lap, idx_lap - 1)
    resize!(V_lap, idx_lap - 1)

    resize!(I_edge, idx_edge - 1)
    resize!(J_edge, idx_edge - 1)
    resize!(V_edge, idx_edge - 1)

    # 10. Construct Final Matrices
    cliqueSampleLap = sparse(I_lap, J_lap, V_lap, n, n)
    new_multiedges = sparse(I_edge, J_edge, V_edge, n, n)

    return cliqueSampleLap, new_multiedges
end

cliqueSampleMultiEdgeSddEffSparseOpt (generic function with 1 method)

In [ ]:
function initMultiedgesSparse(L::SparseMatrixCSC, nr_multi::Int)
    n = size(L, 1)

    # 1. Extract structural non-zeros from L
    # findnz returns the Row indices (I), Column indices (J), and Values (V)
    rows, cols, vals = findnz(L)

    # 2. Filter out the diagonal entries
    # We want indices where row != col AND the value is essentially non-zero
    # (Though typically structural non-zeros in L are actual edges)
    mask = (rows .!= cols) .& (abs.(vals) .> 1e-12)

    edge_rows = rows[mask]
    edge_cols = cols[mask]

    # 3. Create the values vector
    # Every valid edge gets the value 'nr_multi'
    edge_vals = fill(nr_multi, length(edge_rows))

    # 4. Construct the sparse matrix
    # sparse(I, J, V, m, n) creates the matrix efficiently
    multiedges = sparse(edge_rows, edge_cols, edge_vals, n, n)

    return multiedges
end

initMultiedgesSparse (generic function with 1 method)

In [ ]:
function apxCholMultiSDDSparse(L::SparseMatrixCSC, nr_multi::Int)
    n = size(L, 1)

    # 1. Initialize Sparse Structures
    multi_edges = initMultiedgesSparse(L, nr_multi)

    S = copy(L) # Initial sparse copy

    # Initialize triplets for the factor L
    fac_I = Int[]
    fac_J = Int[]
    fac_V = ComplexF64[] # Ensure this is Complex

    sizehint!(fac_I, nnz(L))
    sizehint!(fac_J, nnz(L))
    sizehint!(fac_V, nnz(L))

    for i = 1:(n-1)
        # 2. Extract pivot column
        ai = S[:, i]
        di = S[i, i]

        # In a valid Hermitian Cholesky, diagonal pivots must be real and positive.
        # We take real(di) to ensure type stability and avoid numerical noise (e.g. 1e-18im).
        inv_sqrt_di = 1 / sqrt(real(di))

        # Get nonzero indices and values for column i
        rows = rowvals(S)[nzrange(S, i)]
        vals = nonzeros(S)[nzrange(S, i)]

        for (idx, row) in enumerate(rows)
            # Factor L calculation: L_ki = S_ki / sqrt(S_ii)
            # vals[idx] is S_row,i
            val = vals[idx] * inv_sqrt_di

            push!(fac_I, row)
            push!(fac_J, i)
            push!(fac_V, val)
        end

        # 3. Construct the "Removal Matrix" (M_rem)
        rem_I = Int[]
        rem_J = Int[]
        rem_V = ComplexF64[]

        # Re-iterate the sparse column 'ai' to build the removal updates
        for (idx, row) in enumerate(rows)
            val = vals[idx]

            # 3a. Diagonal Updates
            # For Hermitian SDD, the diagonal dominance is based on magnitude |A_ij|.
            # abs(val) works correctly for Complex numbers (returns modulus).
            push!(rem_I, row); push!(rem_J, row); push!(rem_V, abs(val))

            # 3b. Column i removal (S[row, i])
            # We subtract the value exactly as is
            push!(rem_I, row); push!(rem_J, i); push!(rem_V, val)

            # 3c. Row i removal (S[i, row])
            # HERMITIAN CHANGE: S[i, row] should be conj(S[row, i])
            # Therefore, we must subtract conj(val) to zero it out.
            if row != i
                push!(rem_I, i); push!(rem_J, row); push!(rem_V, conj(val))
            end
        end

        M_rem = sparse(rem_I, rem_J, rem_V, n, n)

        # 4. Sample the Clique
        # Note: Ensure your 'cliqueSample' function returns a Hermitian update
        # (i.e., if it adds edge (u,v) with weight w, it adds (v,u) with weight conj(w)).
        sampled_clique, new_multiedges = cliqueSampleMultiEdgeSddEffSparseOpt(ai, multi_edges[:, i], i)

        # 5. Update Matrices
        S = S - M_rem + sampled_clique
        multi_edges = multi_edges + new_multiedges

        # Handle the final corner case for the last diagonal element
        if i == n - 1
             # S[n,n] is real in Hermitian matrices
             final_val = sqrt(abs(real(S[n, n])))
             push!(fac_I, n); push!(fac_J, n); push!(fac_V, final_val)
        end
    end

    # Construct final sparse factor matrix
    fac = sparse(fac_I, fac_J, fac_V, n, n)
    return fac
end

apxCholMultiSDDSparse (generic function with 1 method)

In [ ]:
function generate_random_sparse_hermitian_sdd(n::Int, density::Float64; strict_dominance::Float64=0.1)
    # 1. Generate random sparse entries for the upper triangle
    # We generate two real sparse matrices for real/imag parts to control structure
    rng_real = sprand(n, n, density)
    rng_imag = sprand(n, n, density)

    # Combine to make complex weights.
    # We strictly mask to the upper triangle (k=1) to define unique edges first.
    U = triu(rng_real + im * rng_imag, 1)

    # 2. Make it Hermitian (A = A')
    # If U contains the upper triangle edges, A becomes the full adjacency matrix.
    A = U + U'

    # 3. Calculate Diagonal Requirements
    # For SDD, |M_ii| >= sum(|M_ij|) for all j != i.
    # We calculate the sum of absolute values of the off-diagonal row entries.
    row_sums = sum(abs, A, dims=2) # Returns an Nx1 matrix

    # 4. Construct the Matrix
    # We set M = D - A (Laplacian-like structure)
    # D must be at least 'row_sums'. We add 'strict_dominance' to ensure it is positive definite.

    # Create the sparse diagonal matrix
    D = spdiagm(0 => vec(row_sums) .+ strict_dominance)

    # M has positive diagonal and negative (complex) off-diagonals
    M = D - A

    return M
end

generate_random_sparse_hermitian_sdd (generic function with 1 method)

In [ ]:
using SparseArrays

"""
    weighted_grid_laplacian(m, n, weights_h, weights_v)

Returns the Laplacian matrix for an m x n grid graph.
- weights_h: (m x n-1) matrix of weights for horizontal edges.
- weights_v: (m-1 x n) matrix of weights for vertical edges.
"""
function weighted_grid_laplacian(m, n, weights_h, weights_v)
    N = m * n
    # We use a Coordinate (COO) format builder for efficiency
    I = Int[]
    J = Int[]
    V = Float64[]

    function add_edge!(u, v, w)
        # Off-diagonal entries: -w
        push!(I, u, v); push!(J, v, u); push!(V, -w, -w)
        # We will handle diagonal entries (degrees) at the end
    end

    # Linear index helper
    idx(r, c) = (c - 1) * m + r

    for r in 1:m
        for c in 1:n
            # Horizontal edges (r, c) -- (r, c+1)
            if c < n
                add_edge!(idx(r, c), idx(r, c+1), weights_h[r, c])
            end
            # Vertical edges (r, c) -- (r+1, c)
            if r < m
                add_edge!(idx(r, c), idx(r+1, c), weights_v[r, c])
            end
        end
    end

    # Create the adjacency-based part of the Laplacian
    L = sparse(I, J, V, N, N)

    # Calculate degrees and set the diagonal
    # The degree of a node is the sum of weights of incident edges.
    # In the Laplacian L = D - A, L[i,i] = sum(abs(L[i,j]) for j != i)
    degrees = abs.(sum(L, dims=2))
    for i in 1:N
        L[i, i] = degrees[i]
    end

    return L
end

# --- Example Usage ---
m, n = 3, 3
wh = rand(m, n-1) # Random horizontal weights
wv = rand(m-1, n) # Random vertical weights

L = weighted_grid_laplacian(m, n, wh, wv)

# Display a small section of the Laplacian
println("Size of Laplacian: ", size(L))
display(L)

Size of Laplacian: (9, 9)


9×9 SparseMatrixCSC{Float64, Int64} with 33 stored entries:
  1.21567   -0.588872    ⋅        …    ⋅          ⋅          ⋅ 
 -0.588872   1.18883   -0.365854       ⋅          ⋅          ⋅ 
   ⋅        -0.365854   0.490646       ⋅          ⋅          ⋅ 
 -0.626794    ⋅          ⋅           -0.609875    ⋅          ⋅ 
   ⋅        -0.234105    ⋅             ⋅        -0.672793    ⋅ 
   ⋅          ⋅        -0.124792  …    ⋅          ⋅        -0.761916
   ⋅          ⋅          ⋅            1.1842    -0.574323    ⋅ 
   ⋅          ⋅          ⋅           -0.574323   1.92477   -0.67765
   ⋅          ⋅          ⋅             ⋅        -0.67765    1.43957

In [ ]:
using SparseArrays

"""
    hermitian_grid_laplacian(m, n, weights_h, weights_v)

Returns a Hermitian Laplacian matrix for an m x n grid.
- weights_h: (m x n-1) Matrix{ComplexF64} - weights for horizontal edges.
- weights_v: (m-1 x n) Matrix{ComplexF64} - weights for vertical edges.
"""
function hermitian_grid_laplacian(m, n, weights_h, weights_v)
    N = m * n
    I = Int[]
    J = Int[]
    V = ComplexF64[]

    # Helper to add Hermitian edge entries
    function add_edge!(u, v, w)
        # L[u,v] = -w, L[v,u] = -conj(w)
        push!(I, u, v)
        push!(J, v, u)
        push!(V, -w, -conj(w))
    end

    idx(r, c) = (c - 1) * m + r

    for r in 1:m
        for c in 1:n
            if c < n
                add_edge!(idx(r, c), idx(r, c+1), weights_h[r, c])
            end
            if r < m
                add_edge!(idx(r, c), idx(r+1, c), weights_v[r, c])
            end
        end
    end

    # Build the off-diagonal sparse matrix
    L = sparse(I, J, V, N, N)

    # For a Hermitian Laplacian, the diagonal entries (degrees)
    # must be real. We sum the magnitudes of the weights.
    # This ensures the matrix is Positive Semi-Definite.
    for i in 1:N
        # Sum of absolute values of off-diagonal elements in row i
        deg = sum(abs.(nonzeros(L[i, :])))
        L[i, i] = deg
    end

    return L
end

# --- Verification ---
m, n = 3, 3
wh = rand(ComplexF64, m, n-1)
wv = rand(ComplexF64, m-1, n)

L = hermitian_grid_laplacian(m, n, wh, wv)
println(matrix(L))
# Check if L is Hermitian: L ≈ L' (adjoint)
is_hermitian = L ≈ L'
println("Is the matrix Hermitian? ", is_hermitian)

# Check eigenvalues (they should all be real for a Hermitian matrix)
using LinearAlgebra
evs = eigvals(Array(L))
println("Smallest eigenvalue (should be ~0): ", minimum(real.(evs)))

LoadError: UndefVarError: `matrix` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
m, n = 25, 50
split = 10
ratios_record = Float64[]
for i in 1:10
  wh = rand(ComplexF64, m, n-1)
  wv = rand(ComplexF64, m-1, n)
  # FIX: density must be a positional argument, and strict_dominance should be a float
  S = hermitian_grid_laplacian(m, n, wh, wv)
  fApxChol = apxCholMultiSDDSparse(S, split) # Second param specifies the number of multiedges
  rel_eigs = eigvals(pinv(Matrix(fApxChol))*Matrix(S)*pinv(Matrix(fApxChol))')
      # The ratio of the smallest to the largest eigenvalue

  println(maximum(abs.(rel_eigs)) / minimum(abs.(rel_eigs)))
end

1.4558856164311818
1.5471982594026965
1.5878239010418513
1.6098064332037831
1.553619093875034
1.6678307644237347
1.5951984292980872
1.5039352788088773
1.4805457294018964
1.5679079870564716


In [ ]:
m, n = 25, 50

for i in 1:10
  wh = rand( m, n-1)
  ah = randn(m,n-1)
  h_new = wh.* exp.(-1im*ah/10000)

  wv = rand(m-1, n) # Corrected dimensions: (m-1) x n
  av = randn(m-1,n) # Corrected dimensions: (m-1) x n
  v_new = wv.* exp.(1im*av/10000)
  # FIX: density must be a positional argument, and strict_dominance should be a float
  S = hermitian_grid_laplacian(m, n, h_new, v_new)
  fApxChol = apxCholMultiSDDSparse(S, 20) # Second param specifies the number of multiedges
  rel_eigs = eigvals(pinv(Matrix(fApxChol))*Matrix(S)*pinv(Matrix(fApxChol))')
      # The ratio of the smallest to the largest eigenvalue

  println(maximum(abs.(rel_eigs)) / minimum(abs.(rel_eigs)))
end

1.6068178150439578
1.8326446596741441
1.6838411551141823
1.682655879232876
1.754006024220491
1.66432847897791
1.8007961339483687
1.7291067906337048
1.7161302749454812
1.5410885093276911


In [ ]:
m, n = 50, 50

for i in 1:10
  wh = rand(ComplexF64, m, n-1)
  wv = rand(ComplexF64, m-1, n)
  # FIX: density must be a positional argument, and strict_dominance should be a float
  S = hermitian_grid_laplacian(m, n, wh, wv)
  fApxChol = apxCholMultiSDDSparse(S, 20) # Second param specifies the number of multiedges
  rel_eigs = eigvals(pinv(Matrix(fApxChol))*Matrix(S)*pinv(Matrix(fApxChol))')
      # The ratio of the smallest to the largest eigenvalue

  println(maximum(abs.(rel_eigs)) / minimum(abs.(rel_eigs)))
end

1.6302892204587527
1.5122577980881224
1.5851071582955905
1.6898069539659542
1.7031307532338695
1.5867438905952664
1.5895609112480666
1.6328367715282348
1.669258232226053
1.5458481536914739


In [ ]:
for i in 1:10
    n = 2000
      # FIX: density must be a positional argument, and strict_dominance should be a float
    S = generate_random_sparse_hermitian_sdd(n, 0.005; strict_dominance=0.000000001)
    fApxChol = apxCholMultiSDDSparse(S, 20) # Second param specifies the number of multiedges
    rel_eigs = eigvals(pinv(Matrix(fApxChol))*Matrix(S)*pinv(Matrix(fApxChol))')
      # The ratio of the smallest to the largest eigenvalue

    println(maximum(abs.(rel_eigs)) / minimum(abs.(rel_eigs)))
end

LoadError: InterruptException:

In [ ]:
for i in 1:10
    n = 2000
      # FIX: density must be a positional argument, and strict_dominance should be a float
    S = generate_random_sparse_hermitian_sdd(n, 0.005; strict_dominance=0.000000001)
    fApxChol = apxCholMultiSDDSparse(S, 20) # Second param specifies the number of multiedges
    rel_eigs = eigvals(pinv(Matrix(fApxChol))*Matrix(S)*pinv(Matrix(fApxChol))')
      # The ratio of the smallest to the largest eigenvalue

    println(maximum(abs.(rel_eigs)) / minimum(abs.(rel_eigs)))
end

LoadError: InterruptException:

In [ ]:
function cliqueSampleHybrid(a::SparseVector{ComplexF64, Int}, multiplicity::SparseVector{Int, Int}, i::Int)
    n = length(a)
    rows = a.nzind
    vals = a.nzval
    m = length(rows)

    abs_vals = abs.(vals)
    ptr_i = searchsortedfirst(rows, i)
    has_i = (ptr_i <= m) && (rows[ptr_i] == i)

    # 1. Original Logic: Self-loop and Ratio
    abs_val_i = has_i ? abs_vals[ptr_i] : 0.0
    sum_abs_all = sum(abs_vals)
    self_loop = abs(2 * abs_val_i - sum_abs_all)

    total_weight = abs_val_i
    ratio = (total_weight > 1e-12) ? self_loop / total_weight : 0.0

    # 2. Pre-allocate Triplet Buffers
    I_lap, J_lap, V_lap = Int[], Int[], ComplexF64[]
    I_edge, J_edge, V_edge = Int[], Int[], Int[]

    # Diagonal Initialization Logic
    for k_idx in 1:m
        if rows[k_idx] != i
            push!(I_lap, rows[k_idx]); push!(J_lap, rows[k_idx]); push!(V_lap, ratio * abs_vals[k_idx])
        end
    end

    # 3. Sampling Setup
    sampling_weights = copy(abs_vals)
    if has_i
        sampling_weights[ptr_i] = self_loop
    end
    wv = Weights(sampling_weights)

    # 4. Optimized Main Loop
    for j_idx in 1:m
        row_j = rows[j_idx]
        if row_j == i || sampling_weights[j_idx] <= 1e-12
            continue
        end

        val_j = vals[j_idx]
        inv_sign_j = conj(val_j / abs_vals[j_idx])
        mult_j = multiplicity[row_j]

        if mult_j > 0
            val_j_scaled = val_j / mult_j
            # Vectorized sampling: much faster than scalar loop
            k_indices = sample(1:m, wv, mult_j)

            for k_idx in k_indices
                row_k = rows[k_idx]
                if row_k != row_j && row_k != i
                    wjk = abs(val_j_scaled * vals[k_idx]) / (abs_vals[j_idx] + abs_vals[k_idx])
                    s = -(vals[k_idx] / abs_vals[k_idx]) * inv_sign_j

                    # Update Lap (Manually build rank-1)
                    push!(I_lap, row_j); push!(J_lap, row_j); push!(V_lap, wjk)
                    push!(I_lap, row_k); push!(J_lap, row_k); push!(V_lap, wjk)
                    push!(I_lap, row_j); push!(J_lap, row_k); push!(V_lap, wjk * conj(s))
                    push!(I_lap, row_k); push!(J_lap, row_j); push!(V_lap, wjk * s)

                    # Update Edges
                    push!(I_edge, row_j); push!(J_edge, row_k); push!(V_edge, 1)
                    push!(I_edge, row_k); push!(J_edge, row_j); push!(V_edge, 1)
                end
            end
        end
    end

    return sparse(I_lap, J_lap, V_lap, n, n), sparse(I_edge, J_edge, V_edge, n, n)
end

cliqueSampleHybrid (generic function with 1 method)

In [ ]:
function apxCholMultiSDDSparse_Hybrid(L::SparseMatrixCSC{ComplexF64, Int}, nr_multi::Int)
    n = size(L, 1)
    multi_edges = initMultiedgesSparse(L, nr_multi)
    S = copy(L)

    fac_I, fac_J, fac_V = Int[], Int[], ComplexF64[]

    for i = 1:(n-1)
        # 1. Pivot Column Extraction
        ai = S[:, i]
        di = S[i, i]
        inv_sqrt_di = 1.0 / sqrt(real(di))

        # 2. Build Factor L column
        rows = ai.nzind
        vals = ai.nzval
        for (idx, row) in enumerate(rows)
            push!(fac_I, row); push!(fac_J, i); push!(fac_V, vals[idx] * inv_sqrt_di)
        end

        # 3. Original Schur Complement Logic
        # This part ensures correctness by mimicking S_noi
        S_noi = copy(S)
        # Subtract absolute magnitudes from diagonal
        for (idx, row) in enumerate(rows)
            S_noi[row, row] -= abs(vals[idx])
        end
        # Zero out pivot row/column
        S_noi[:, i] .= 0
        S_noi[i, :] .= 0
        dropzeros!(S_noi)

        # 4. Sample the clique using the optimized sampler
        sampled_clique, new_multiedges = cliqueSampleHybrid(ai, multi_edges[:, i], i)

        # 5. Update Matrices
        S = S_noi + sampled_clique
        multi_edges = multi_edges + new_multiedges

        if i == n - 1
             push!(fac_I, n); push!(fac_J, n); push!(fac_V, sqrt(abs(real(S[n, n]))))
        end
    end

    return sparse(fac_I, fac_J, fac_V, n, n)
end

apxCholMultiSDDSparse_Hybrid (generic function with 1 method)

In [ ]:
function cliqueSampleHybridNew(a::SparseVector{ComplexF64, Int}, multiplicity::SparseVector{Int, Int}, diag_idx::Int)
    n = length(a)
    rows = a.nzind
    vals = a.nzval
    m = length(rows)

    # 1. Identify pivot and calculate self-loop logic
    abs_vals = abs.(vals)
    ptr_i = searchsortedfirst(rows, diag_idx)
    has_i = (ptr_i <= m) && (rows[ptr_i] == diag_idx)

    abs_val_i = has_i ? abs_vals[ptr_i] : 0.0
    sum_abs_all = sum(abs_vals)
    self_loop = abs(2 * abs_val_i - sum_abs_all)

    # 2. Ratio for diagonal updates
    ratio = (abs_val_i > 1e-12) ? self_loop / abs_val_i : 0.0

    I_lap, J_lap, V_lap = Int[], Int[], ComplexF64[]
    I_edge, J_edge, V_edge = Int[], Int[], Int[]

    # Diagonal Initial Updates
    for k_idx in 1:m
        if rows[k_idx] != diag_idx
            push!(I_lap, rows[k_idx]); push!(J_lap, rows[k_idx]); push!(V_lap, ratio * abs_vals[k_idx])
        end
    end

    # 3. Setup Weights and Vectorized Sampling
    sampling_weights = copy(abs_vals)
    if has_i; sampling_weights[ptr_i] = self_loop; end
    wv = Weights(sampling_weights)

    for j_ptr in 1:m
        row_j = rows[j_ptr]
        if row_j == diag_idx || sampling_weights[j_ptr] <= 1e-12
            continue
        end

        mult_j = multiplicity[row_j]
        if mult_j > 0
            val_j = vals[j_ptr]
            inv_sign_j = conj(val_j / abs_vals[j_ptr])
            val_j_scaled = val_j / mult_j

            # Efficiently sample indices for cliques
            k_ptrs = sample(1:m, wv, mult_j)

            for k_ptr in k_ptrs
                row_k = rows[k_ptr]
                if row_k != row_j && row_k != diag_idx
                    wjk = abs(val_j_scaled * vals[k_ptr]) / (abs_vals[j_ptr] + abs_vals[k_ptr])
                    s = -(vals[k_ptr] / abs_vals[k_ptr]) * inv_sign_j

                    # Accumulate rank-1 update triplets
                    push!(I_lap, row_j); push!(J_lap, row_j); push!(V_lap, wjk)
                    push!(I_lap, row_k); push!(J_lap, row_k); push!(V_lap, wjk)
                    push!(I_lap, row_j); push!(J_lap, row_k); push!(V_lap, wjk * conj(s))
                    push!(I_lap, row_k); push!(J_lap, row_j); push!(V_lap, wjk * s)

                    # Accumulate multi-edge count triplets
                    push!(I_edge, row_j); push!(J_edge, row_k); push!(V_edge, 1)
                    push!(I_edge, row_k); push!(J_edge, row_j); push!(V_edge, 1)
                end
            end
        end
    end
    return I_lap, J_lap, V_lap, I_edge, J_edge, V_edge
end

cliqueSampleHybridNew (generic function with 1 method)

In [ ]:
function apxCholMultiSDDSparse_NoCopy(L::SparseMatrixCSC{ComplexF64, Int}, nr_multi::Int)
    n = size(L, 1)
    multi_edges = initMultiedgesSparse(L, nr_multi)
    S = copy(L) # Initial copy outside the loop

    fac_I, fac_J, fac_V = Int[], Int[], ComplexF64[]
    sizehint!(fac_I, nnz(L) * 2)

    for i = 1:(n-1)
        # S is now the current active submatrix. Pivot is always node 1.
        ai_rel = S[:, 1]
        di = real(S[1, 1])
        inv_sqrt_di = 1.0 / sqrt(di)

        # 1. Map relative indices to global indices for the Factor matrix
        rows_rel = ai_rel.nzind
        vals_rel = ai_rel.nzval
        for (idx, r_rel) in enumerate(rows_rel)
            global_row = r_rel + i - 1
            push!(fac_I, global_row); push!(fac_J, i); push!(fac_V, vals_rel[idx] * inv_sqrt_di)
        end

        # 2. Get Clique and Edge triplets relative to node 1
        cI, cJ, cV, eI, eJ, eV = cliqueSampleHybridNew(ai_rel, multi_edges[:, 1], 1)

        # 3. Apply the S_noi diagonal logic: S[j,j] -= |ai[j]|
        # We collect these as triplets for a single merge operation.
        diag_I, diag_V = Int[], ComplexF64[]
        for (idx, r_rel) in enumerate(rows_rel)
            if r_rel > 1
                push!(diag_I, r_rel - 1); push!(diag_V, -abs(vals_rel[idx]))
            end
        end

        # 4. Slicing: This is the critical replacement for copy(S) and zeroing-out
        S_sub = S[2:end, 2:end]
        multi_sub = multi_edges[2:end, 2:end]

        # Determine size of next submatrix
        m_dim = n - i

        # 5. Merge all updates in a single sparse operation
        # We slice the clique/edge results to match the submatrix size.
        S = S_sub +
            sparse(diag_I, diag_I, diag_V, m_dim, m_dim) +
            sparse(filter_sub(cI, cJ, cV, m_dim)...)

        multi_edges = multi_sub + sparse(filter_sub(eI, eJ, eV, m_dim)...)

        if i == n - 1
             push!(fac_I, n); push!(fac_J, n); push!(fac_V, sqrt(abs(real(S[1, 1]))))
        end
    end

    return sparse(fac_I, fac_J, fac_V, n, n)
end

# Helper to filter triplets for the submatrix (indices > 1)
function filter_sub(I, J, V, dim)
    mask = (I .> 1) .& (J .> 1)
    return I[mask] .- 1, J[mask] .- 1, V[mask], dim, dim
end

filter_sub (generic function with 1 method)

In [ ]:
for i in 1:10
n = 2000
  # FIX: density must be a positional argument, and strict_dominance should be a float
  S = generate_random_sparse_hermitian_sdd(n, 0.005; strict_dominance=0.000000001)
  fApxChol = apxCholMultiSDDSparse_NoCopy(S, 200) # Second param specifies the number of multiedges
  rel_eigs = eigvals(Matrix(S)-Matrix(fApxChol)*Matrix(fApxChol)')
  # The ratio of the smallest to the largest eigenvalue

  println(maximum(abs.(rel_eigs)) / maximum(abs.(eigvals(Matrix(S)))))
end

LoadError: InterruptException:

In [ ]:
function getCliqueTriplets_InPlace!(I_lap, J_lap, V_lap, I_edge, J_edge, V_edge,
                                    a::SparseVector{ComplexF64, Int},
                                    multiplicity_col::SparseVector{Int, Int},
                                    i::Int)
    rows = a.nzind
    vals = a.nzval
    m = length(rows)

    # 1. Identify pivot and calculate self-loop logic
    abs_vals = abs.(vals)
    ptr_i = searchsortedfirst(rows, i)
    has_i = (ptr_i <= m) && (rows[ptr_i] == i)

    abs_val_i = has_i ? abs_vals[ptr_i] : 0.0
    sum_abs_all = sum(abs_vals)
    self_loop = abs(2 * abs_val_i - sum_abs_all)

    # 2. Ratio for diagonal updates
    ratio = (abs_val_i > 1e-12) ? self_loop / abs_val_i : 0.0

    # Diagonal Initial Updates - Added directly to the Lap buffers
    for k_idx in 1:m
        if rows[k_idx] != i
            push!(I_lap, rows[k_idx]); push!(J_lap, rows[k_idx]); push!(V_lap, ratio * abs_vals[k_idx])
        end
    end

    # 3. Setup Weights and Vectorized Sampling
    sampling_weights = copy(abs_vals)
    if has_i; sampling_weights[ptr_i] = self_loop; end
    wv = Weights(sampling_weights)

    for j_ptr in 1:m
        row_j = rows[j_ptr]
        if row_j == i || sampling_weights[j_ptr] <= 1e-12
            continue
        end

        # Lookup multiplicity in the sparse column
        mult_j = multiplicity_col[row_j]

        if mult_j > 0
            val_j = vals[j_ptr]
            inv_sign_j = conj(val_j / abs_vals[j_ptr])
            val_j_scaled = val_j / mult_j

            # Efficiently sample indices for cliques (returns Integers)
            k_ptrs = sample(1:m, wv, mult_j)

            for k_ptr in k_ptrs
                row_k = rows[k_ptr]
                if row_k != row_j && row_k != i
                    wjk = abs(val_j_scaled * vals[k_ptr]) / (abs_vals[j_ptr] + abs_vals[k_ptr])
                    s = -(vals[k_ptr] / abs_vals[k_ptr]) * inv_sign_j

                    # Accumulate rank-1 update triplets
                    push!(I_lap, row_j); push!(J_lap, row_j); push!(V_lap, wjk)
                    push!(I_lap, row_k); push!(J_lap, row_k); push!(V_lap, wjk)
                    push!(I_lap, row_j); push!(J_lap, row_k); push!(V_lap, wjk * conj(s))
                    push!(I_lap, row_k); push!(J_lap, row_j); push!(V_lap, wjk * s)

                    # Accumulate multi-edge count triplets
                    push!(I_edge, row_j); push!(J_edge, row_k); push!(V_edge, 1)
                    push!(I_edge, row_k); push!(J_edge, row_j); push!(V_edge, 1)
                end
            end
        end
    end
end

getCliqueTriplets_InPlace! (generic function with 1 method)

In [ ]:
function apxCholMultiSDDSparse_Ultra(L::SparseMatrixCSC{ComplexF64, Int}, nr_multi::Int)
    n = size(L, 1)

    # Initialize multi_edges as a Dict or an adjacency list for O(1) updates
    # SparseMatrix + SparseMatrix is the bottleneck; Dict is better for high-frequency updates.
    multi_edges = initMultiedgesSparse(L, nr_multi)
    S = copy(L)

    fac_I, fac_J, fac_V = Int[], Int[], ComplexF64[]
    sizehint!(fac_I, nnz(L) * 2)

    # Pre-allocate update buffers to reuse memory
    upd_I, upd_J, upd_V = Int[], Int[], ComplexF64[]
    edge_I, edge_J, edge_V = Int[], Int[], Int[]

    for i = 1:(n-1)
        # 1. Fast Column Access
        # Use direct CSC pointers to avoid slicing/copying the column
        col_ptr = S.colptr[i]:(S.colptr[i+1]-1)
        if isempty(col_ptr) continue end

        rows = view(S.rowval, col_ptr)
        vals = view(S.nzval, col_ptr)

        # 2. Extract Pivot and Build Factor L
        d_idx = searchsortedfirst(rows, i)
        di = real(vals[d_idx])
        inv_sqrt_di = 1.0 / sqrt(di)

        @inbounds for k in 1:length(rows)
            push!(fac_I, rows[k]); push!(fac_J, i); push!(fac_V, vals[k] * inv_sqrt_di)
        end

        # 3. Targeted Updates instead of Matrix Addition
        # We only collect updates for nodes > i (the future submatrix)
        empty!(upd_I); empty!(upd_J); empty!(upd_V)
        empty!(edge_I); empty!(edge_J); empty!(edge_V)

        # Original Logic: Subtract |ai| from diagonal
        for k in 1:length(rows)
            if rows[k] > i
                push!(upd_I, rows[k]); push!(upd_J, rows[k]); push!(upd_V, -abs(vals[k]))
            end
        end

        # 4. Sampling with i-th column SparseVector
        # We reuse the hybrid sampler but pass our pre-allocated buffers
        getCliqueTriplets_InPlace!(upd_I, upd_J, upd_V, edge_I, edge_J, edge_V,
                                   S[:, i], multi_edges[:, i], i)

        # 5. The "Golden" Optimization: Efficient Merge
        # Instead of S = S_sub + ..., we update the multi_edges and S
        # only on the non-zero structure that changed.
        if !isempty(upd_I)
            # Create a small update matrix
            S += sparse(upd_I, upd_J, upd_V, n, n)
            multi_edges += sparse(edge_I, edge_J, edge_V, n, n)
        end

        # Explicitly remove the pivot row/column from the structure
        # This is much faster than copy(S) because it doesn't move all the data
        # but we must eventually handle the last pivot.
    end

    return sparse(fac_I, fac_J, fac_V, n, n)
end

apxCholMultiSDDSparse_Ultra (generic function with 1 method)

In [ ]:
for i in 1:1
n = 2000
  # FIX: density must be a positional argument, and strict_dominance should be a float
  S = generate_random_sparse_hermitian_sdd(n, 0.005; strict_dominance=0.000000001)
  fApxChol = apxCholMultiSDDSparse_Ultra(S, 200) # Second param specifies the number of multiedges
  rel_eigs = eigvals(Matrix(S)-Matrix(fApxChol)*Matrix(fApxChol)')
  # The ratio of the smallest to the largest eigenvalue

  println(maximum(abs.(rel_eigs)) / maximum(abs.(eigvals(Matrix(S)))))
end

LoadError: InterruptException:

In [ ]:
# Convert SparseMatrixCSC to Vector of Dicts
function sparse_to_adj(L::SparseMatrixCSC{T, Int}) where T
    n = size(L, 1)
    adj = [Dict{Int, T}() for _ in 1:n]
    rows = rowvals(L)
    vals = nonzeros(L)
    for j in 1:n
        for i_ptr in nzrange(L, j)
            i = rows[i_ptr]
            adj[i][j] = vals[i_ptr]
        end
    end
    return adj
end

# Efficiently get a "SparseVector" like view from a Dict for the sampler
function get_col_from_adj(adj, col_idx)
    d = adj[col_idx]
    # We sort keys to maintain consistency with SparseVector logic
    keys_sorted = sort(collect(keys(d)))
    vals_sorted = [d[k] for k in keys_sorted]
    return SparseVector(length(adj), keys_sorted, vals_sorted)
end

get_col_from_adj (generic function with 1 method)

In [ ]:
function getCliqueTriplets_Symmetric!(I_lap, J_lap, V_lap, I_edge, J_edge, V_edge,
                                       a::SparseVector{ComplexF64, Int},
                                       multiplicity_col::SparseVector{Int, Int},
                                       i::Int)
    rows = a.nzind
    vals = a.nzval
    m = length(rows)

    abs_vals = abs.(vals)
    ptr_i = searchsortedfirst(rows, i)
    has_i = (ptr_i <= m) && (rows[ptr_i] == i)

    abs_val_i = has_i ? abs_vals[ptr_i] : 0.0
    sum_abs_all = sum(abs_vals)
    self_loop = abs(2 * abs_val_i - sum_abs_all)

    ratio = (abs_val_i > 1e-12) ? self_loop / abs_val_i : 0.0

    # Diagonal Initial Updates
    for k_idx in 1:m
        r = rows[k_idx]
        if r != i
            push!(I_lap, r); push!(J_lap, r); push!(V_lap, ratio * abs_vals[k_idx])
        end
    end

    sampling_weights = copy(abs_vals)
    if has_i; sampling_weights[ptr_i] = self_loop; end
    wv = Weights(sampling_weights)

    for j_ptr in 1:m
        row_j = rows[j_ptr]
        if row_j == i || sampling_weights[j_ptr] <= 1e-12
            continue
        end

        mult_j = multiplicity_col[row_j]
        if mult_j > 0
            val_j = vals[j_ptr]
            inv_sign_j = conj(val_j / abs_vals[j_ptr])
            val_j_scaled = val_j / mult_j
            k_ptrs = sample(1:m, wv, mult_j)

            for k_ptr in k_ptrs
                row_k = rows[k_ptr]
                if row_k != row_j && row_k != i
                    wjk = abs(val_j_scaled * vals[k_ptr]) / (abs_vals[j_ptr] + abs_vals[k_ptr])
                    s = -(vals[k_ptr] / abs_vals[k_ptr]) * inv_sign_j

                    # Update Lap: Push both (j,k) and (k,j) explicitly
                    push!(I_lap, row_j); push!(J_lap, row_j); push!(V_lap, wjk)
                    push!(I_lap, row_k); push!(J_lap, row_k); push!(V_lap, wjk)

                    # Off-diagonal Hermitian pair
                    push!(I_lap, row_j); push!(J_lap, row_k); push!(V_lap, wjk * conj(s))
                    push!(I_lap, row_k); push!(J_lap, row_j); push!(V_lap, wjk * s)

                    # Multi-edges are symmetric by nature in SDD
                    push!(I_edge, row_j); push!(J_edge, row_k); push!(V_edge, 1)
                    push!(I_edge, row_k); push!(J_edge, row_j); push!(V_edge, 1)
                end
            end
        end
    end
end

getCliqueTriplets_Symmetric! (generic function with 1 method)

In [ ]:
function apxCholMultiSDDSparse_Mirror(L::SparseMatrixCSC{ComplexF64, Int}, nr_multi::Int)
    n = size(L, 1)

    # 1. Initialize structures
    S_adj = sparse_to_adj(L)
    multi_adj = sparse_to_adj(initMultiedgesSparse(L, nr_multi))

    fac_I, fac_J, fac_V = Int[], Int[], ComplexF64[]
    sizehint!(fac_I, nnz(L) * 2)

    for i = 1:(n-1)
        # --- STEP 1: Snapshot current state (matches ai = S[:,i]) ---
        # We need the column as a SparseVector for the sampler
        ai_dict = S_adj[i]
        if isempty(ai_dict) || !haskey(ai_dict, i) continue end

        # Ensure ai is captured EXACTLY as it is in the original S[:,i]
        rows_ai = sort(collect(keys(ai_dict)))
        vals_ai = [ai_dict[r] for r in rows_ai]
        ai_sparse = SparseVector(n, rows_ai, vals_ai)

        mi_sparse = get_col_from_adj(multi_adj, i)

        # --- STEP 2: Factor Calculation ---
        di = real(ai_dict[i])
        inv_sqrt_di = 1.0 / sqrt(di)
        for r in rows_ai
            push!(fac_I, r); push!(fac_J, i); push!(fac_V, ai_dict[r] * inv_sqrt_di)
        end

        # --- STEP 3: Create S_noi (The "Cleaning" Phase) ---
        # Mimic: S_noi -= Diagonal(abs.(ai))
        for r in rows_ai
            S_adj[r][r] -= abs(ai_dict[r])
        end

        # Mimic: S_noi[:, i] .= 0 and S_noi[i, :] .= 0
        # We clear the pivot column i
        empty!(S_adj[i])
        # We clear the pivot row i from all other columns
        for k in 1:n
            if k != i && haskey(S_adj[k], i)
                delete!(S_adj[k], i)
            end
        end

        # --- STEP 4: Sample and Add Clique ---
        # Use the hybrid sampler but collect triplets to update S_adj
        cI, cJ, cV, eI, eJ, eV = cliqueSampleHybrid_Triplets(ai_sparse, mi_sparse, i)

        # Update S_adj (S = S_noi + sampled_clique)
        for k in 1:length(cI)
            r, c, v = cI[k], cJ[k], cV[k]
            # Matrix addition logic
            S_adj[c][r] = get(S_adj[c], r, 0.0im) + v
        end

        # Update multi_edges
        for k in 1:length(eI)
            r, c, v = eI[k], eJ[k], eV[k]
            multi_adj[c][r] = get(multi_adj[c], r, 0) + v
        end

        # Handle final corner case
        if i == n - 1
             final_val = sqrt(abs(real(get(S_adj[n], n, 0.0im))))
             push!(fac_I, n); push!(fac_J, n); push!(fac_V, final_val)
        end
    end

    return sparse(fac_I, fac_J, fac_V, n, n)
end

# Revised Sampler to match original weight and sign logic exactly
function cliqueSampleHybrid_Triplets(a, multiplicity, i)
    n = length(a)
    rows = a.nzind
    vals = a.nzval
    m = length(rows)

    abs_vals = abs.(vals)
    ptr_i = searchsortedfirst(rows, i)
    abs_val_i = abs_vals[ptr_i]

    self_loop = abs(2 * abs_val_i - sum(abs_vals))
    ratio = (abs_val_i > 1e-12) ? self_loop / abs_val_i : 0.0

    I_lap, J_lap, V_lap = Int[], Int[], ComplexF64[]
    I_edge, J_edge, V_edge = Int[], Int[], Int[]

    # Diagonal Logic: matches original loop
    for k_idx in 1:m
        if rows[k_idx] != i
            push!(I_lap, rows[k_idx]); push!(J_lap, rows[k_idx]); push!(V_lap, ratio * abs_vals[k_idx])
        end
    end

    sampling_weights = copy(abs_vals)
    sampling_weights[ptr_i] = self_loop
    wv = Weights(sampling_weights)

    for j_ptr in 1:m
        row_j = rows[j_ptr]
        if row_j == i || sampling_weights[j_ptr] <= 1e-12 continue end

        mult_j = multiplicity[row_j]
        if mult_j > 0
            val_j = vals[j_ptr]
            inv_sign_j = conj(val_j / abs_vals[j_ptr])
            val_j_scaled = val_j / mult_j
            k_ptrs = sample(1:m, wv, mult_j)

            for k_ptr in k_ptrs
                row_k = rows[k_ptr]
                if row_k != row_j && row_k != i
                    wjk = abs(val_j_scaled * vals[k_ptr]) / (abs_vals[j_ptr] + abs_vals[k_ptr])
                    s = -(vals[k_ptr] / abs_vals[k_ptr]) * inv_sign_j

                    # Symmetrized triplets
                    push!(I_lap, row_j); push!(J_lap, row_j); push!(V_lap, wjk)
                    push!(I_lap, row_k); push!(J_lap, row_k); push!(V_lap, wjk)
                    push!(I_lap, row_j); push!(J_lap, row_k); push!(V_lap, wjk * conj(s))
                    push!(I_lap, row_k); push!(J_lap, row_j); push!(V_lap, wjk * s)

                    push!(I_edge, row_j); push!(J_edge, row_k); push!(V_edge, 1)
                    push!(I_edge, row_k); push!(J_edge, row_j); push!(V_edge, 1)
                end
            end
        end
    end
    return I_lap, J_lap, V_lap, I_edge, J_edge, V_edge
end

cliqueSampleHybrid_Triplets (generic function with 1 method)

In [ ]:
for i in 1:10
n = 2000
  # FIX: density must be a positional argument, and strict_dominance should be a float
  S = generate_random_sparse_hermitian_sdd(n, 0.005; strict_dominance=0.000000001)
  fApxChol = apxCholMultiSDDSparse_Mirror(S, 200) # Second param specifies the number of multiedges
  rel_eigs = eigvals(Matrix(S)-Matrix(fApxChol)*Matrix(fApxChol)')
  # The ratio of the smallest to the largest eigenvalue

  println(maximum(abs.(rel_eigs)) / maximum(abs.(eigvals(Matrix(S)))))
end

LoadError: InterruptException:

In [ ]:
n=2000
S = generate_random_sparse_hermitian_sdd(n, 0.005; strict_dominance=0.000000001)
rel_eigs = eigvals(Matrix(S))

LoadError: InterruptException: